In [1]:
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus

In [2]:
df = pd.read_excel("BASE.xlsx")
df.head()

,clicodigo,clitipoidentificacion,clinombres,cliapellidos,clitipo,CLIFechaNacimiento,CLIFechaCreacion,CLICelular,CLIEmailPrincipal,ALMCodigo,CLICodClasificacion,CLICampo4,CLIHabeas
0,1013260820,CC,MARIANA,DIAZ,1.0,2023-01-19 00:00:00,2023-01-15 14:27:54.743,3134495621,MARIDIAZD08@HOTMAIL.COM,318,5.000000e+10,NaN,0
1,1017218266,CC,DIEGO,REYES GONZALEZ,1.0,1994-04-11 00:00:00,2021-06-06 19:21:43.147,3052270509,DIREYESGO30@GMAIL.COM,215,4.000000e+10,NaN,1
2,1019024338,CC,MONICA PAOLA,GARZON GIL,1.0,1988-03-25 00:00:00,2021-06-09 12:36:44.413,3144874565,MONISOFIA2014@GMAIL.COM,308,3.000000e+10,NaN,1
3,1020742092,CC,ALEXANDRA,PIÑEROS PINEDA,1.0,1989-02-25 00:00:00,2021-06-03 15:05:31.950,3194843765,MAYITO0225TWILIGHT@HOTAIL.COM,310,5.000000e+10,NaN,1
4,1035231968,CC,LIZET DAYANA,PINTO RESTREPO,1.0,1996-01-23 00:00:00,2021-06-04 13:05:20.353,3105047389,LIZETH1623@HOTMAIL.COM,215,1.000000e+10,NaN,1


In [3]:
# Normalizar columnas si tienen espacios o nombres extraños
df.columns = df.columns.str.strip().str.lower()

In [3]:
print(df.columns.tolist())

['clicodigo', 'clitipoidentificacion', 'clinombres', 'cliapellidos', 'clitipo', 'CLIFechaNacimiento', 'CLIFechaCreacion', 'CLICelular', 'CLIEmailPrincipal', 'ALMCodigo', 'CLICodClasificacion', 'CLICampo4', 'CLIHabeas']


In [4]:

# Eliminar registros sin correo ni apellido
df = df.dropna(subset=['clicodigo'])

In [5]:
# Limpiar columna de celular
df['CLICelular'] = df['CLICelular'].astype(str)

# Eliminar espacios, paréntesis, guiones, etc.
df['CLICelular'] = df['CLICelular'].str.replace(r'[^\d+]', '', regex=True)

# Eliminar prefijo +57 si existe
df['CLICelular'] = df['CLICelular'].str.replace(r'^\+?57', '', regex=True)

# Asegurarse que queda solo con números
df['CLICelular'] = df['CLICelular'].str.replace(r'\D', '', regex=True)

# Filtrar: exactamente 10 dígitos y que empiece con 3
df = df[df['CLICelular'].str.match(r'^3\d{9}$')]

# Eliminar celulares claramente inválidos
celulares_invalidos = {
    '1111111111', '1234567890', '0000000000', '9999999999', '2222222222'
}
df = df[~df['CLICelular'].isin(celulares_invalidos)]

In [6]:

# Validar que el correo tenga una estructura básica válida
#df = df[df['EMAIL'].str.contains(r'^[\w\.-]+@[\w\.-]+\.\w+$', regex=True)]

df = df[df['CLIEmailPrincipal'].str.contains(r'^[\w\.-]+@[\w\.-]+\.\w+$', regex=True, na=False)]


In [7]:
# Resetear el índice
df = df.reset_index(drop=True)

In [8]:
# Guardar archivo limpio
df.to_excel('CumpleañosClientes.xlsx', index=False)